<a href="https://colab.research.google.com/github/AsimaZaheer/Conference-Research-Repo/blob/main/version%203.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
from google.colab import drive
drive.mount('/content/drive')

import os
import zipfile

ZIP_PATH = "/content/drive/MyDrive/WESAD.zip"
EXTRACT_PATH = "/content/WESAD_extracted"

if not os.path.exists(ZIP_PATH):
    raise FileNotFoundError(
        f"WESAD.zip nahi mili: {ZIP_PATH}\n"
        "Apni ZIP file ka correct Google Drive path set karo."
    )

os.makedirs(EXTRACT_PATH, exist_ok=True)

with zipfile.ZipFile(ZIP_PATH, "r") as zip_ref:
    zip_ref.extractall(EXTRACT_PATH)

print("Dataset extract ho gaya!")

pkl_files = []

for root, dirs, files in os.walk(EXTRACT_PATH):
    for file in files:
        if file.endswith(".pkl") and file.startswith("S"):
            pkl_files.append(os.path.join(root, file))

print("PKL files found:", len(pkl_files))
print("Example files:", pkl_files[:5])

if not pkl_files:
    raise FileNotFoundError(
        "ZIP ke andar subject PKL files nahi milin. "
        "ZIP structure check karo."
    )

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Dataset extract ho gaya!
PKL files found: 15
Example files: ['/content/WESAD_extracted/WESAD/S6/S6.pkl', '/content/WESAD_extracted/WESAD/S17/S17.pkl', '/content/WESAD_extracted/WESAD/S7/S7.pkl', '/content/WESAD_extracted/WESAD/S5/S5.pkl', '/content/WESAD_extracted/WESAD/S10/S10.pkl']


In [10]:
import pickle
import numpy as np
import pandas as pd

from scipy.signal import resample_poly
from collections import Counter
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score
)

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV, GroupKFold
from sklearn.ensemble import RandomForestClassifier

In [11]:

from scipy.signal import butter, sosfiltfilt, find_peaks

FS_EDA = 4
FS_BVP = 64
FS_LABEL = 700

WINDOW_SECONDS = 60
STRIDE_SECONDS = 5

WINDOW_SIZE = WINDOW_SECONDS * FS_EDA
STRIDE_SIZE = STRIDE_SECONDS * FS_EDA

BVP_RATIO = FS_BVP // FS_EDA  # 16


# -----------------------------------------
# 1. EDA: statistical + tonic/phasic features
# -----------------------------------------
def extract_eda_features(signal):
    signal = np.asarray(signal, dtype=float).reshape(-1)

    if len(signal) < 20 or not np.all(np.isfinite(signal)):
        raise ValueError("Invalid EDA window")

    # Basic statistical features
    basic = [
        np.mean(signal),
        np.std(signal),
        np.min(signal),
        np.max(signal),
        np.ptp(signal)
    ]

    # Tonic component: slow changes in EDA
    tonic_sos = butter(
        2, 0.05, btype="lowpass",
        fs=FS_EDA, output="sos"
    )
    tonic = sosfiltfilt(tonic_sos, signal)

    # Phasic component: faster changes
    phasic = signal - tonic

    tonic_slope = np.polyfit(
        np.arange(len(tonic)) / FS_EDA,
        tonic, 1
    )[0]

    phasic_peaks, _ = find_peaks(
        phasic,
        distance=FS_EDA,
        prominence=max(np.std(phasic) * 0.5, 1e-8)
    )

    physiological = [
        np.mean(tonic),
        np.std(tonic),
        tonic_slope,
        np.std(phasic),
        np.mean(np.abs(phasic)),
        np.max(phasic),
        len(phasic_peaks) / WINDOW_SECONDS
    ]

    return basic + physiological


# -----------------------------------------
# 2. BVP: pulse-related features at 64 Hz
# -----------------------------------------
def extract_bvp_features(signal):
    signal = np.asarray(signal, dtype=float).reshape(-1)

    if len(signal) < 100 or not np.all(np.isfinite(signal)):
        raise ValueError("Invalid BVP window")

    # Band-pass filter to retain pulse-related frequencies
    bvp_sos = butter(
        3, [0.5, 8.0], btype="bandpass",
        fs=FS_BVP, output="sos"
    )
    filtered = sosfiltfilt(bvp_sos, signal)

    # Detect candidate pulse peaks
    peaks, _ = find_peaks(
        filtered,
        distance=int(0.3 * FS_BVP),
        prominence=max(np.std(filtered) * 0.3, 1e-8)
    )

    # Basic statistics of filtered BVP
    basic = [
        np.mean(filtered),
        np.std(filtered),
        np.min(filtered),
        np.max(filtered),
        np.ptp(filtered)
    ]

    # Pulse intervals and estimated heart rate
    if len(peaks) >= 2:
        intervals = np.diff(peaks) / FS_BVP
        heart_rates = 60.0 / intervals

        # Exclude implausible intervals for these features
        valid = (
            (intervals >= 60.0 / 200.0) &
            (intervals <= 60.0 / 40.0)
        )

        valid_intervals = intervals[valid]
        valid_hr = heart_rates[valid]

        if len(valid_hr) > 0:
            hr_mean = np.mean(valid_hr)
            hr_std = np.std(valid_hr)
            ibi_mean = np.mean(valid_intervals)
            ibi_std = np.std(valid_intervals)
        else:
            hr_mean, hr_std, ibi_mean, ibi_std = [0.0] * 4

        # Peak-to-peak pulse amplitude
        amplitudes = filtered[peaks[1:]] - filtered[peaks[:-1]]

        amp_mean = np.mean(amplitudes)
        amp_std = np.std(amplitudes)

    else:
        hr_mean, hr_std, ibi_mean, ibi_std = [0.0] * 4
        amp_mean, amp_std = 0.0, 0.0

    physiological = [
        hr_mean,
        hr_std,
        ibi_mean,
        ibi_std,
        amp_mean,
        amp_std,
        len(peaks) / WINDOW_SECONDS
    ]

    return basic + physiological


# -----------------------------------------
# 3. Load signals; preserve BVP at 64 Hz
# -----------------------------------------
def load_subject(file_path):
    with open(file_path, "rb") as f:
        data = pickle.load(f, encoding="latin1")

    subject_name = os.path.basename(file_path).replace(".pkl", "")
    wrist = data["signal"]["wrist"]

    eda = np.asarray(wrist["EDA"]).reshape(-1)
    bvp_64 = np.asarray(wrist["BVP"]).reshape(-1)
    labels = np.asarray(data["label"]).reshape(-1)

    # Map each 4 Hz EDA sample to the corresponding
    # approximate label sample at 700 Hz
    label_indices = (
        np.arange(len(eda)) * (FS_LABEL / FS_EDA)
    ).astype(int)

    valid = label_indices < len(labels)
    eda = eda[:np.sum(valid)]
    label_indices = label_indices[valid]
    labels_4hz = labels[label_indices]

    # Align duration without downsampling BVP
    n = min(
        len(eda),
        len(labels_4hz),
        len(bvp_64) // BVP_RATIO
    )

    return (
        subject_name,
        eda[:n],
        bvp_64[:n * BVP_RATIO],
        labels_4hz[:n]
    )


# -----------------------------------------
# 4. Windowing and feature extraction
# -----------------------------------------
def make_windows(eda, bvp, labels, subject):
    rows = []

    n = min(len(eda), len(labels), len(bvp) // BVP_RATIO)

    for start in range(
        0, n - WINDOW_SIZE + 1, STRIDE_SIZE
    ):
        end = start + WINDOW_SIZE

        eda_window = eda[start:end]

        # Matching 60-second BVP segment at 64 Hz
        bvp_window = bvp[
            start * BVP_RATIO : end * BVP_RATIO
        ]

        label_window = labels[start:end]

        # Retain only pure Baseline or pure Stress windows
        counts = Counter(label_window.tolist())

        if counts.get(1, 0) == len(label_window):
            target = 0
        elif counts.get(2, 0) == len(label_window):
            target = 1
        else:
            continue

        try:
            eda_features = extract_eda_features(eda_window)
            bvp_features = extract_bvp_features(bvp_window)
        except ValueError:
            continue

        rows.append({
            "subject": subject,
            "label": target,
            "eda_features": eda_features,
            "bvp_features": bvp_features
        })

    return rows


In [12]:
all_rows = []

for file_path in sorted(pkl_files):
    subject = os.path.basename(file_path).replace(".pkl", "")

    # Sirf S2-S17, S12 excluded
    try:
        subject_num = int(subject[1:])
    except (ValueError, IndexError):
        continue

    if subject_num < 2 or subject_num > 17 or subject_num == 12:
        continue

    print("Processing:", subject)

    subject, eda, bvp, labels = load_subject(file_path)

    rows = make_windows(
        eda=eda,
        bvp=bvp,
        labels=labels,
        subject=subject
    )

    all_rows.extend(rows)

    print("  Valid windows:", len(rows))

if not all_rows:
    raise ValueError(
        "Koi valid window nahi bani. Dataset paths, labels aur "
        "signal lengths check karo."
    )

print("\nTotal valid windows:", len(all_rows))
print("Subjects:", sorted(set(r["subject"] for r in all_rows)))

print("\nClass distribution:")
print(pd.Series([r["label"] for r in all_rows])
      .map({0: "Baseline", 1: "Stress"})
      .value_counts())

Processing: S10
  Valid windows: 357
Processing: S11
  Valid windows: 348
Processing: S13
  Valid windows: 345
Processing: S14
  Valid windows: 347
Processing: S15
  Valid windows: 349
Processing: S16
  Valid windows: 347
Processing: S17
  Valid windows: 357
Processing: S2
  Valid windows: 328
Processing: S3
  Valid windows: 332
Processing: S4
  Valid windows: 334
Processing: S5
  Valid windows: 345
Processing: S6
  Valid windows: 342
Processing: S7
  Valid windows: 341
Processing: S8
  Valid windows: 344
Processing: S9
  Valid windows: 341

Total valid windows: 5157
Subjects: ['S10', 'S11', 'S13', 'S14', 'S15', 'S16', 'S17', 'S2', 'S3', 'S4', 'S5', 'S6', 'S7', 'S8', 'S9']

Class distribution:
Baseline    3342
Stress      1815
Name: count, dtype: int64


In [13]:

X_eda = np.array([
    r["eda_features"] for r in all_rows
], dtype=float)

X_fusion = np.array([
    r["eda_features"] + r["bvp_features"]
    for r in all_rows
], dtype=float)

y = np.array([r["label"] for r in all_rows])
groups = np.array([r["subject"] for r in all_rows])

print("EDA feature matrix:", X_eda.shape)
print("EDA+BVP feature matrix:", X_fusion.shape)
print("Labels:", y.shape)

# Same LOSO folds for all three models

logo = LeaveOneGroupOut()
results = []

for fold, (train_idx, test_idx) in enumerate(
    logo.split(X_eda, y, groups), start=1
):
    X_train, X_test = X_eda[train_idx], X_eda[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    test_subject = groups[test_idx][0]

    # 1. Majority-class baseline
    majority_model = DummyClassifier(strategy="most_frequent")
    majority_model.fit(X_train, y_train)
    pred_majority = majority_model.predict(X_test)

    # 2. Logistic Regression
    logistic_model = make_pipeline(
        StandardScaler(),
        LogisticRegression(
            max_iter=2000,
            class_weight="balanced",
            random_state=42
        )
    )
    logistic_model.fit(X_train, y_train)
    pred_logistic = logistic_model.predict(X_test)


    # 3. Tuned Random Forest
    train_groups = groups[train_idx]
    n_train_subjects = len(np.unique(train_groups))

    if n_train_subjects < 3:
        raise ValueError(
            "GroupKFold ke liye kam az kam 3 training subjects chahiye."
        )

    inner_cv = GroupKFold(n_splits=3)

    param_grid = {
        "n_estimators": [100, 200],
        "max_depth": [None, 10, 20],
        "min_samples_leaf": [1, 3],
        "max_features": ["sqrt"]
    }

    search = GridSearchCV(
        estimator=RandomForestClassifier(
            random_state=42,
            class_weight="balanced",
            n_jobs=1
        ),
        param_grid=param_grid,
        scoring="balanced_accuracy",
        cv=inner_cv,
        n_jobs=-1,
        refit=True,
        error_score="raise"
    )

    search.fit(
        X_train,
        y_train,
        groups=train_groups
    )

    rf_model = search.best_estimator_
    pred_rf = rf_model.predict(X_test)

    print(
        f"{test_subject} best parameters: {search.best_params_}"
    )

    # Evaluate all models on the same held-out subject
    for model_name, predictions in [
        ("Majority Baseline", pred_majority),
        ("Logistic Regression", pred_logistic),
        ("Random Forest", pred_rf)
    ]:
        results.append({
            "Subject": test_subject,
            "Model": model_name,
            "Accuracy": accuracy_score(y_test, predictions),
            "Balanced Accuracy": balanced_accuracy_score(
                y_test, predictions
            ),
            "Macro F1": f1_score(
                y_test, predictions,
                average="macro",
                zero_division=0
            )
        })

results_df = pd.DataFrame(results)

# Results for every held-out subject
display(
    results_df.sort_values(["Subject", "Model"])
    .round(4)
)

# Mean and standard deviation across LOSO folds
summary_df = (
    results_df.groupby("Model")[
        ["Accuracy", "Balanced Accuracy", "Macro F1"]
    ]
    .agg(["mean", "std"])
    .round(4)
)

print("\nLOSO SUMMARY (mean ± standard deviation):")
display(summary_df)


EDA feature matrix: (5157, 12)
EDA+BVP feature matrix: (5157, 24)
Labels: (5157,)
S10 best parameters: {'max_depth': 10, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'n_estimators': 200}
S11 best parameters: {'max_depth': 10, 'max_features': 'sqrt', 'min_samples_leaf': 3, 'n_estimators': 100}
S13 best parameters: {'max_depth': 10, 'max_features': 'sqrt', 'min_samples_leaf': 3, 'n_estimators': 100}
S14 best parameters: {'max_depth': 10, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'n_estimators': 100}
S15 best parameters: {'max_depth': 10, 'max_features': 'sqrt', 'min_samples_leaf': 3, 'n_estimators': 200}
S16 best parameters: {'max_depth': 10, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'n_estimators': 100}
S17 best parameters: {'max_depth': 10, 'max_features': 'sqrt', 'min_samples_leaf': 3, 'n_estimators': 100}
S2 best parameters: {'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 3, 'n_estimators': 200}
S3 best parameters: {'max_depth': None, 'max_features': 'sqrt

,Subject,Model,Accuracy,Balanced Accuracy,Macro F1
1,S10,Logistic Regression,0.8039,0.7475,0.7639
0,S10,Majority Baseline,0.6275,0.5000,0.3855
2,S10,Random Forest,0.8571,0.8113,0.8320
4,S11,Logistic Regression,0.8534,0.8862,0.8505
3,S11,Majority Baseline,0.6437,0.5000,0.3916
5,S11,Random Forest,0.5862,0.6732,0.5812
7,S13,Logistic Regression,0.6435,0.7216,0.6427
6,S13,Majority Baseline,0.6493,0.5000,0.3937
8,S13,Random Forest,0.7362,0.7969,0.7359
10,S14,Logistic Regression,0.6686,0.5417,0.4882



LOSO SUMMARY (mean ± standard deviation):


Accuracy         Balanced Accuracy         Macro F1  \
                        mean     std              mean     std     mean   
Model                                                                     
Logistic Regression   0.7918  0.1344            0.7807  0.1475   0.7567   
Majority Baseline     0.6482  0.0109            0.5000  0.0000   0.3933   
Random Forest         0.8142  0.1195            0.7967  0.1176   0.7852   

                             
                        std  
Model                        
Logistic Regression  0.1680  
Majority Baseline    0.0040  
Random Forest        0.1537

In [14]:
summary_df.to_csv(
    "/content/WESAD_overall_results.csv",
    index=False
)

subject_df.to_csv(
    "/content/WESAD_subject_results.csv",
    index=False
)

print("Results saved successfully.")

from google.colab import files

files.download("/content/WESAD_overall_results.csv")
files.download("/content/WESAD_subject_results.csv")

Results saved successfully.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>